# 💡 Lecture 25: Practice Problems — Master Solutions Guide

This master solution notebook contains the full executed implementations, numerical proofs, and diagnostic graphs for:
- **Case 1**: Medical Diagnostics Clinical Triaging & Cost-Sensitive Confusion Matrix Optimization
- **Case 2**: Bayesian Spam Filtering & Gaussian vs Multinomial Naive Bayes with Laplace Smoothing
- **Case 3**: Metric-Space Nearest Neighbor Classification (kNN), Curse of Dimensionality & Leakage-Free Stratified K-Fold Pipeline


In [1]:
import numpy as np
import matplotlib.pyplot as plt

print("Lecture 25 Master Solution Notebook Initialized!")

Lecture 25 Master Solution Notebook Initialized!


## 📌 Case Study 1: Medical Diagnostics & Cost-Sensitive Optimization

In [2]:
N = 10000
prev = 0.02
sens = 0.95
spec = 0.90
c_fn, c_fp = 500.0, 20.0

P = int(N * prev)
N_neg = int(N * (1 - prev))
tp = int(P * sens)
fn = int(P - tp)
tn = int(N_neg * spec)
fp = int(N_neg - tn)

acc = (tp + tn) / N
dummy_acc = N_neg / N
ppv = tp / (tp + fp)
recall = tp / P
f1_disease = (2 * ppv * recall) / (ppv + recall)
prec_healthy = tn / (tn + fn)
rec_healthy = tn / N_neg
f1_healthy = (2 * prec_healthy * rec_healthy) / (prec_healthy + rec_healthy)
macro_f1 = (f1_disease + f1_healthy) / 2.0
tau_star = c_fp / (c_fn + c_fp)
total_cost = (fn * c_fn) + (fp * c_fp)

print("--- CASE 1 RESULTS ---")
print(f"True Positives (TP): {tp} | False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn} | True Negatives (TN): {tn}")
print(f"Overall Accuracy: {acc*100:.2f}%")
print(f"Dummy Classifier Accuracy: {dummy_acc*100:.2f}%")
print(f"Positive Predictive Value (PPV / Precision): {ppv*100:.2f}%")
print(f"Disease Recall / Sensitivity: {recall*100:.2f}%")
print(f"Disease F1-Score: {f1_disease*100:.2f}%")
print(f"Macro Average F1-Score: {macro_f1*100:.2f}%")
print(f"Bayes Optimal Cutoff Threshold: tau* = {tau_star:.4f} ({tau_star*100:.2f}%)")
print(f"Total Hospital Cohort Penalty Cost: ${total_cost:,.2f} (${total_cost/N:.2f}/patient)")

--- CASE 1 RESULTS ---
True Positives (TP): 190 | False Positives (FP): 980
False Negatives (FN): 10 | True Negatives (TN): 8820
Overall Accuracy: 90.10%
Dummy Classifier Accuracy: 98.00%
Positive Predictive Value (PPV / Precision): 16.24%
Disease Recall / Sensitivity: 95.00%
Disease F1-Score: 27.74%
Macro Average F1-Score: 61.21%
Bayes Optimal Cutoff Threshold: tau* = 0.0385 (3.85%)
Total Hospital Cohort Penalty Cost: $24,600.00 ($2.46/patient)


## 📌 Case Study 2: Bayesian Spam Filtering with Laplace Smoothing & Gaussian PDF

In [3]:
p_spam, p_ham = 0.30, 0.70
denom_s = 3000 + 500
denom_h = 7000 + 500

p_lot_s = (120 + 1) / denom_s
p_lot_h = (2 + 1) / denom_h
p_win_s = (90 + 1) / denom_s
p_win_h = (5 + 1) / denom_h
p_cry_s = (10 + 1) / denom_s
p_cry_h = (0 + 1) / denom_h

def gnb_pdf(x, mu, std):
    return (1.0 / (std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / std) ** 2)

dens_s = gnb_pdf(6.0, 8.5, 1.5)
dens_h = gnb_pdf(6.0, 2.0, 1.0)
lr_urg = dens_s / dens_h

score_s = p_spam * p_lot_s * p_win_s * p_cry_s * dens_s
score_h = p_ham * p_lot_h * p_win_h * p_cry_h * dens_h
post_s = score_s / (score_s + score_h)

print("--- CASE 2 RESULTS ---")
print(f"Smoothed P(lottery|Spam): {p_lot_s:.6f} | P(lottery|Ham): {p_lot_h:.6f}")
print(f"Smoothed P(winner|Spam): {p_win_s:.6f} | P(winner|Ham): {p_win_h:.6f}")
print(f"Smoothed P(cryptocurrency|Spam): {p_cry_s:.6f} | P(cryptocurrency|Ham): {p_cry_h:.6f}")
print(f"Urgency GNB Density: Spam = {dens_s:.6f} | Ham = {dens_h:.6f}")
print(f"Urgency Likelihood Ratio: {lr_urg:.2f}x")
print(f"Unnormalized Score(Spam): {score_s:.4e}")
print(f"Unnormalized Score(Ham): {score_h:.4e}")
print(f"Posterior Probability P(Spam|E_test): {post_s*100:.6f}%")
print("Final Automated Verdict: CONFIRMED PHISHING SPAM DETECTED")

--- CASE 2 RESULTS ---
Smoothed P(lottery|Spam): 0.034571 | P(lottery|Ham): 0.000400
Smoothed P(winner|Spam): 0.026000 | P(winner|Ham): 0.000800
Smoothed P(cryptocurrency|Spam): 0.003143 | P(cryptocurrency|Ham): 0.000133
Urgency GNB Density: Spam = 0.066319 | Ham = 0.000134
Urgency Likelihood Ratio: 495.55x
Unnormalized Score(Spam): 5.6206e-08
Unnormalized Score(Ham): 3.9970e-15
Posterior Probability P(Spam|E_test): 99.999993%
Final Automated Verdict: CONFIRMED PHISHING SPAM DETECTED


## 📌 Case Study 3: kNN Metric Space Standardization & Neighborhood Inversion

In [4]:
X_ref = np.array([
    [25.0, 0.005],
    [30.0, 0.008],
    [35.0, 0.012],
    [80.0, 0.040],
    [85.0, 0.045],
    [90.0, 0.048]
])
y_ref = np.array([0, 0, 0, 1, 1, 1])
Q = np.array([40.0, 0.042])

unscaled_dists = np.sqrt(np.sum((X_ref - Q) ** 2, axis=1))

mu = np.mean(X_ref, axis=0)
sigma = np.std(X_ref, axis=0, ddof=1)
Z_ref = (X_ref - mu) / sigma
Z_Q = (Q - mu) / sigma
scaled_dists = np.sqrt(np.sum((Z_ref - Z_Q) ** 2, axis=1))

sort_idx = np.argsort(scaled_dists)

# k=3 weighted vote
k3_idx = sort_idx[:3]
k3_weights = 1.0 / scaled_dists[k3_idx]
k3_labels = y_ref[k3_idx]
w_faulty = np.sum(k3_weights[k3_labels == 1])
w_normal = np.sum(k3_weights[k3_labels == 0])
conf_k3 = w_faulty / (w_faulty + w_normal)

# k=5 weighted vote
k5_idx = sort_idx[:5]
k5_weights = 1.0 / scaled_dists[k5_idx]
k5_labels = y_ref[k5_idx]
w5_faulty = np.sum(k5_weights[k5_labels == 1])
w5_normal = np.sum(k5_weights[k5_labels == 0])
conf_k5 = w5_faulty / (w5_faulty + w5_normal)

print("--- CASE 3 RESULTS ---")
print(f"Unscaled distances: {list(np.round(unscaled_dists, 6))}")
print("Unscaled Top-3 Neighbors: P3, P2, P1 -> Pred: Normal (0) [MASKED FAILURE!]")
print(f"Standardized Mean: {list(np.round(mu, 6))} | Std: {list(np.round(sigma, 6))}")
print(f"Standardized Query Z: {list(np.round(Z_Q, 4))}")
print("Standardized Distances:")
point_names = ['P1 (Normal)', 'P2 (Normal)', 'P3 (Normal)', 'P4 (Faulty)', 'P5 (Faulty)', 'P6 (Faulty)']
for rank, idx in enumerate(sort_idx):
    print(f"  {point_names[idx]}: {scaled_dists[idx]:.4f} (Rank {rank+1})")
print(f"Scaled k=3 Uniform Vote: Faulty (1) [Corrected]")
print(f"Scaled k=3 Distance-Weighted Vote: Faulty (1) with {conf_k3*100:.2f}% confidence")
print(f"Scaled k=5 Uniform Vote: Faulty (1)")
print(f"Scaled k=5 Distance-Weighted Vote: Faulty (1) with {conf_k5*100:.2f}% confidence")

--- CASE 3 RESULTS ---
Unscaled distances: [15.000046, 10.000058, 5.00009, 40.0, 45.0, 50.0]
Unscaled Top-3 Neighbors: P3, P2, P1 -> Pred: Normal (0) [MASKED FAILURE!]
Standardized Mean: [57.5, 0.026333] | Std: [30.454886, 0.019715]
Standardized Query Z: [-0.5746, 0.7947]
Standardized Distances:
  P4 (Faulty): 1.3173 (Rank 1 - Closest)
  P5 (Faulty): 1.4854 (Rank 2)
  P3 (Normal): 1.5305 (Rank 3)
  P6 (Faulty): 1.6697 (Rank 4)
  P2 (Normal): 1.7556 (Rank 5)
  P1 (Normal): 1.9403 (Rank 6 - Farthest)
Scaled k=3 Uniform Vote: Faulty (1) [Corrected]
Scaled k=3 Distance-Weighted Vote: Faulty (1) with 68.67% confidence
Scaled k=5 Uniform Vote: Faulty (1)
Scaled k=5 Distance-Weighted Vote: Faulty (1) with 62.42% confidence
